# What Makes Coffee Score High?

An exploration of 1,300 specialty coffees from 36 countries — examining
what drives quality, how fermentation shapes flavor, and why some of the
industry's most common beliefs don't hold up in the data.

In [31]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"

from pathlib import Path
from utils.data import load_cqi

# Use the most enriched dataset available
soil_path = Path("..") / "data" / "processed" / "cqi_with_climate_soil.csv"
climate_path = Path("..") / "data" / "processed" / "cqi_with_climate.csv"
if soil_path.exists():
    df = pd.read_csv(soil_path)
    df.columns = df.columns.str.strip().str.lower().str.replace(".", "_", regex=False).str.replace(" ", "_")
    has_climate = True
    has_soil = True
elif climate_path.exists():
    df = pd.read_csv(climate_path)
    df.columns = df.columns.str.strip().str.lower().str.replace(".", "_", regex=False).str.replace(" ", "_")
    has_climate = True
    has_soil = False
else:
    df = load_cqi()
    has_climate = False
    has_soil = False

# Load Cup of Excellence data if available
coe_path = Path("..") / "data" / "raw" / "cup_of_excellence.csv"
has_coe = coe_path.exists()
if has_coe:
    coe = pd.read_csv(coe_path, low_memory=False)
    coe["score"] = pd.to_numeric(coe["score"], errors="coerce")
    # Filter to valid cupping scores (85-100 range for CoE)
    coe = coe[(coe["score"] >= 80) & (coe["score"] <= 100)].copy()

cupping_attrs = ["aroma", "flavor", "aftertaste", "acidity", "body", "balance"]
main_methods = ["Washed / Wet", "Natural / Dry", "Pulped natural / honey", "Semi-washed / Semi-pulped"]
method_colors = {
    "Washed / Wet": "#4a90d9",
    "Natural / Dry": "#d94a4a",
    "Pulped natural / honey": "#d9a54a",
    "Semi-washed / Semi-pulped": "#4ad9a5",
}

## The Dataset

Every coffee submitted to the [Coffee Quality Institute](https://www.coffeeinstitute.org/)
goes through a standardized evaluation called **cupping**. Trained graders score each coffee
on six sensory attributes — aroma, flavor, aftertaste, acidity, body, and balance — which
are summed into a **Total Cup Points** score.

This dataset contains 1,311 Arabica coffees, each with cupping scores, origin details
(country, region, farm, altitude), the coffee variety, and crucially — the
**processing method**, which determines how the coffee cherry is fermented after harvest.

In [32]:
# Dataset overview
overview = pd.DataFrame({
    "Metric": ["Total coffees", "Countries", "Processing methods", "Varieties",
               "Score range", "Mean score"],
    "Value": [
        f"{len(df):,}",
        str(df["country_of_origin"].nunique()),
        str(df["processing_method"].nunique()),
        str(df["variety"].nunique()),
        f"{df['total_cup_points'].min():.1f} – {df['total_cup_points'].max():.1f}",
        f"{df['total_cup_points'].mean():.1f}",
    ]
})

fig = go.Figure(data=go.Table(
    header=dict(values=list(overview.columns), align="left"),
    cells=dict(values=[overview[c] for c in overview.columns], align="left"),
))
fig.update_layout(height=250, margin=dict(t=10, b=10))
fig.show()

---

## 1. Flavor and Aftertaste Drive Everything

> **Observed in the data.** These correlations are computed directly from 1,300 samples.

Not all cupping attributes contribute equally to a coffee's total score.
**Flavor** (r=0.88) and **aftertaste** (r=0.87) are by far the strongest predictors.
A coffee that tastes great and finishes clean will score high — even if it has
average body or unremarkable acidity.

**Body is the most independent attribute** (r=0.78). It correlates least with
the other scores, meaning a coffee can have exceptional aroma and flavor but
only average mouthfeel. This makes body the most interesting dimension for
distinguishing processing methods — and it's exactly where we see
fermentation's fingerprint.

In [33]:
# What drives total score?
corrs = df[cupping_attrs + ["total_cup_points"]].corr()["total_cup_points"].drop("total_cup_points")
corrs = corrs.sort_values(ascending=True)

colors = ["#d4a574" if v < 0.85 else "#8b5e3c" for v in corrs.values]

fig = px.bar(
    x=corrs.values,
    y=[c.replace("_", " ").title() for c in corrs.index],
    orientation="h",
    labels={"x": "Correlation with Total Score (r)", "y": ""},
    text=[f"{v:.2f}" for v in corrs.values],
)
fig.update_traces(marker_color=colors)
fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    showlegend=False,
    height=350,
)
fig.show()

In [34]:
# How attributes relate to each other
corr_matrix = df[cupping_attrs].corr()

fig = px.imshow(
    corr_matrix,
    x=[c.replace("_", " ").title() for c in cupping_attrs],
    y=[c.replace("_", " ").title() for c in cupping_attrs],
    color_continuous_scale="YlOrBr",
    text_auto=".2f",
    labels={"color": "r"},
    zmin=0.6, zmax=1.0,
)
fig.update_layout(height=450)
fig.show()

---

## 2. Fermentation Matters More Than You'd Think

> **Observed in the data.** The score differences below are computed directly
> from group averages. However, we cannot claim processing *causes* higher
> scores — other factors (origin, variety, farm quality) are not controlled for.

After a coffee cherry is picked, the fruit must be removed before the bean
can be dried and roasted. *How* the fruit is removed — the **processing method** —
determines how much fermentation occurs:

- **Washed**: fruit is removed mechanically, then the bean is soaked briefly.
  Minimal fermentation. Produces clean, bright cups.
- **Natural**: the whole cherry is dried intact on raised beds for weeks.
  Extended fermentation. Produces fruity, heavy-bodied cups.
- **Honey**: a middle ground — some fruit pulp is left on during drying.
- **Semi-washed**: a brief fermentation followed by mechanical washing.

A common industry view is that processing affects flavor *character* but not
overall *quality*. With 1,300 samples, we see that's not quite right.
**More fermentation correlates with slightly higher total scores:**

In [35]:
# Average score by processing method
main_methods = ["Washed / Wet", "Natural / Dry", "Pulped natural / honey", "Semi-washed / Semi-pulped"]
method_df = df[df["processing_method"].isin(main_methods)].copy()

method_stats = method_df.groupby("processing_method").agg(
    avg_score=("total_cup_points", "mean"),
    count=("total_cup_points", "count"),
).reset_index()
method_stats = method_stats.sort_values("avg_score", ascending=True)

fig = px.bar(
    method_stats,
    x="avg_score",
    y="processing_method",
    orientation="h",
    text=method_stats.apply(lambda r: f"{r['avg_score']:.1f}  ({int(r['count'])} samples)", axis=1),
    labels={"avg_score": "Average Total Cup Points", "processing_method": ""},
    color_discrete_sequence=["#8b5e3c"],
)
fig.update_layout(
    xaxis=dict(range=[80, 84]),
    height=300,
    showlegend=False,
)
fig.show()

The gap is modest — less than a point between washed and honey — but it's
consistent and directional. Methods that involve more contact between the bean
and the fruit (and therefore more fermentation) tend to produce higher-scoring coffees.

Note: Honey processing has only 14 samples, so its lead should be taken cautiously.
Washed (812 samples) and Natural (251) are far more robust.

---

## 3. How Fermentation Shapes the Cup

> **Observed in the data.** The attribute-level differences are directly measured.
> The *interpretation* that fermentation causes higher body is an inference —
> it's consistent with food science, but this dataset doesn't isolate fermentation
> from other variables that differ across processing methods.

The total score tells one story, but the individual attributes tell a richer one.
Each processing method produces a distinct **cupping profile** — a signature
pattern across the six attributes.

In [36]:
# Radar chart — cupping profile by processing method
method_df = df[df["processing_method"].isin(main_methods)].copy()
method_avg = method_df.groupby("processing_method")[cupping_attrs].mean()

fig = go.Figure()
for method in main_methods:
    if method in method_avg.index:
        vals = method_avg.loc[method]
        labels = [c.replace("_", " ").title() for c in cupping_attrs]
        fig.add_trace(go.Scatterpolar(
            r=vals.values.tolist() + [vals.values[0]],
            theta=labels + [labels[0]],
            name=method,
            fill="toself",
            opacity=0.4,
            line=dict(color=method_colors.get(method)),
        ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[7.2, 7.7])),
    height=500,
)
fig.show()

**Natural processed coffees push outward on body** (7.60 vs 7.49 for washed) —
the extended fermentation adds mouthfeel and richness. They also edge ahead on
flavor and aroma, likely from the fruit sugars that caramelize during drying.

**Washed coffees are tighter and more uniform** — their profile is the smallest
polygon, trading body and sweetness for consistency.

The box plots below show the full distribution, not just averages. Notice how
naturals have wider spreads — more fermentation means more variability.
When it works, it works beautifully. When it doesn't, the scores drop further.

In [37]:
# Score distributions by processing method
melted = method_df.melt(
    id_vars=["processing_method"],
    value_vars=cupping_attrs,
    var_name="attribute",
    value_name="score",
)
melted["attribute"] = melted["attribute"].str.replace("_", " ").str.title()

fig = px.box(
    melted,
    x="attribute",
    y="score",
    color="processing_method",
    labels={"attribute": "", "score": "Score", "processing_method": "Processing"},
    color_discrete_map=method_colors,
)
fig.update_layout(boxmode="group", height=450)
fig.show()

---

## 4. The Altitude Myth

> **Observed in the data.** The weak correlation is a direct measurement
> across 1,080 samples with altitude data.

Ask anyone in specialty coffee what makes a great coffee and "high altitude"
will come up quickly. The logic makes sense: cooler temperatures at elevation
slow cherry maturation, allowing more complex sugars to develop.

But in this dataset of 1,080 coffees with altitude data, **the correlation
between altitude and cup score is near zero** (r=0.11).

In [38]:
# Altitude vs cup score
alt_df = df.dropna(subset=["altitude_mean_meters", "total_cup_points"])
alt_df = alt_df[alt_df["altitude_mean_meters"] < 5000]

fig = px.scatter(
    alt_df,
    x="altitude_mean_meters",
    y="total_cup_points",
    color="processing_method",
    trendline="ols",
    hover_data=["farm_name", "country_of_origin", "region"],
    labels={"altitude_mean_meters": "Altitude (m)", "total_cup_points": "Total Cup Points"},
    opacity=0.5,
    color_discrete_map=method_colors,
)
fig.update_layout(height=500)
fig.show()

This doesn't mean altitude is irrelevant — it means **altitude alone is a poor
predictor**. It's likely a proxy for the things that actually matter:

- **Temperature** — which altitude approximates, but imprecisely
- **Humidity and rainfall** — which affect drying and fermentation
- **Soil composition** — which varies independently of elevation

A farm at 1,200m in humid Colombia and a farm at 1,200m in dry Ethiopia
will produce very different coffees. Altitude tells you none of that.
To understand what's really going on, we need direct climate measurements.

---

## 5. The World of Coffee, by Origin

> **Observed in the data.** Country averages and distributions are direct measurements.
> Explanations for *why* countries differ (terroir, selection bias, cultural practices)
> are inferences.

Some countries consistently produce higher-scoring coffees than others.
**Ethiopia leads by a wide margin** — averaging 85.5, nearly 2.5 points
ahead of second-place Colombia (83.1).

In [39]:
# Top countries by average score
top_countries = df["country_of_origin"].value_counts().head(10).index
country_df = df[df["country_of_origin"].isin(top_countries)]

country_order = country_df.groupby("country_of_origin")["total_cup_points"].mean().sort_values(ascending=False).index

fig = px.box(
    country_df,
    x="country_of_origin",
    y="total_cup_points",
    color="processing_method",
    hover_data=["farm_name", "variety"],
    labels={"country_of_origin": "", "total_cup_points": "Total Cup Points"},
    category_orders={"country_of_origin": list(country_order)},
    color_discrete_map=method_colors,
)
fig.update_layout(height=500)
fig.show()

A few things stand out:

- **Ethiopia's lead is real but may be amplified by selection bias.** Ethiopian
  coffees submitted to CQI may skew toward exceptional lots — the average
  Ethiopian coffee farm may not score 85+.
- **Honduras has extreme variance** (std=11.65). Some coffees score in the 80s,
  others in the 60s — likely data entry errors rather than genuine quality spread.
- **Brazil is almost entirely natural processed** — visible in the chart above.
  It's the world's largest producer and the birthplace of natural processing.
- **Countries cluster by processing preference**, which makes it hard to
  separate the effect of origin from the effect of processing without
  controlling for both.

In [40]:
# World map — average score by country
import pycountry

def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

country_stats = df.groupby("country_of_origin").agg(
    avg_score=("total_cup_points", "mean"),
    sample_count=("total_cup_points", "count"),
).reset_index()
country_stats["iso3"] = country_stats["country_of_origin"].map(to_iso3)
country_stats = country_stats.dropna(subset=["iso3"])

fig = px.choropleth(
    country_stats,
    locations="iso3",
    locationmode="ISO-3",
    color="avg_score",
    hover_name="country_of_origin",
    hover_data=["sample_count"],
    color_continuous_scale="YlOrBr",
    labels={"avg_score": "Avg Score", "sample_count": "Samples"},
)
fig.update_layout(
    geo=dict(showframe=False, projection_type="natural earth"),
    height=450,
    margin=dict(t=10, b=10),
)
fig.show()

---

## 6. Climate — The Hidden Variable

> **Observed in the data.** The climate values and processing method clustering
> are directly measured. Climate data comes from NASA POWER, filtered to each
> country's harvest months.

Altitude is a proxy. What actually matters is the climate a coffee experiences
during harvest and processing. We geocoded all 1,300+ farms and pulled
monthly temperature, humidity, and rainfall data from
[NASA POWER](https://power.larc.nasa.gov/), filtered to each country's
harvest window.

**On its own, climate barely predicts total score** — temperature correlates
at r=0.01, humidity at r=0.07. But that's the wrong question. The right
question is: **does climate interact with processing method?**

In [41]:
# Where do different processing methods happen, climatically?
climate_df = df[df["processing_method"].isin(main_methods)].dropna(subset=["t2m", "rh2m"])

fig = px.scatter(
    climate_df,
    x="t2m",
    y="rh2m",
    color="processing_method",
    hover_data=["farm_name", "country_of_origin", "total_cup_points"],
    labels={
        "t2m": "Avg Temperature During Harvest (°C)",
        "rh2m": "Avg Relative Humidity (%)",
        "processing_method": "Processing",
    },
    opacity=0.5,
    color_discrete_map=method_colors,
)
fig.update_layout(height=500)
fig.show()

Processing methods cluster climatically. **Natural processing happens in warmer,
drier conditions** (avg 22.5°C, 69% humidity), while **washed coffees come
from cooler, more humid regions** (20.2°C, 70%). This isn't random — farmers
choose methods that suit their climate.

But here's where it gets interesting: **the same processing method performs
differently depending on climate.**

---

## 7. The Climate-Fermentation Interaction

> **Observed in the data.** The overall correlations (r=-0.23 for naturals/temp,
> r=0.23 for washed/rainfall) are directly computed. **However, the within-country
> analysis below reveals these aggregate numbers are partly driven by which
> countries dominate each processing method — not purely by climate.**

When we split by processing method and look at how climate relates to
cup score within each method, a pattern appears in the aggregate:

**Natural processed coffees score lower in hotter climates** (r=-0.23 with
temperature). **Washed coffees score higher in wetter climates** (r=0.23
with rainfall).

In [42]:
# Temperature vs score — split by processing method
natural_df = df[df["processing_method"] == "Natural / Dry"].dropna(subset=["t2m", "total_cup_points"])
washed_df = df[df["processing_method"] == "Washed / Wet"].dropna(subset=["t2m", "total_cup_points"])


fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Natural: cooler = better (r=-0.23)",
    "Washed: temperature doesn't matter (r=0.10)",
])

fig.add_trace(go.Scatter(
    x=natural_df["t2m"], y=natural_df["total_cup_points"],
    mode="markers", marker=dict(color="#d94a4a", opacity=0.5),
    name="Natural", showlegend=False,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=washed_df["t2m"], y=washed_df["total_cup_points"],
    mode="markers", marker=dict(color="#4a90d9", opacity=0.5),
    name="Washed", showlegend=False,
), row=1, col=2)

# Add trendlines
for col_idx, (sub_df, color) in enumerate([(natural_df, "#d94a4a"), (washed_df, "#4a90d9")], 1):
    z = np.polyfit(sub_df["t2m"], sub_df["total_cup_points"], 1)
    x_line = np.linspace(sub_df["t2m"].min(), sub_df["t2m"].max(), 50)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode="lines", line=dict(color=color, width=2),
        showlegend=False,
    ), row=1, col=col_idx)

fig.update_xaxes(title_text="Harvest Temperature (°C)", row=1, col=1)
fig.update_xaxes(title_text="Harvest Temperature (°C)", row=1, col=2)
fig.update_yaxes(title_text="Total Cup Points", row=1, col=1)
fig.update_layout(height=450)
fig.show()

In [43]:
# Rainfall vs score — split by processing method
natural_rain = df[df["processing_method"] == "Natural / Dry"].dropna(subset=["prectotcorr", "total_cup_points"])
washed_rain = df[df["processing_method"] == "Washed / Wet"].dropna(subset=["prectotcorr", "total_cup_points"])

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Natural: rainfall doesn't matter (r=-0.00)",
    "Washed: wetter = better (r=0.23)",
])

fig.add_trace(go.Scatter(
    x=natural_rain["prectotcorr"], y=natural_rain["total_cup_points"],
    mode="markers", marker=dict(color="#d94a4a", opacity=0.5),
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=washed_rain["prectotcorr"], y=washed_rain["total_cup_points"],
    mode="markers", marker=dict(color="#4a90d9", opacity=0.5),
    showlegend=False,
), row=1, col=2)

for col_idx, (sub_df, color) in enumerate([(natural_rain, "#d94a4a"), (washed_rain, "#4a90d9")], 1):
    z = np.polyfit(sub_df["prectotcorr"], sub_df["total_cup_points"], 1)
    x_line = np.linspace(sub_df["prectotcorr"].min(), sub_df["prectotcorr"].max(), 50)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode="lines", line=dict(color=color, width=2),
        showlegend=False,
    ), row=1, col=col_idx)

fig.update_xaxes(title_text="Harvest Rainfall (mm/day)", row=1, col=1)
fig.update_xaxes(title_text="Harvest Rainfall (mm/day)", row=1, col=2)
fig.update_yaxes(title_text="Total Cup Points", row=1, col=1)
fig.update_layout(height=450)
fig.show()

### But does it hold within countries?

> **This is the critical test.** The aggregate correlations could be an artifact
> of country composition — Ethiopia is cool and scores high, Brazil is warm
> and scores lower. If the pattern disappears when we look within a single
> country, it's not really about climate.

The within-country picture is **mixed:**

**Naturals and temperature:**
- **Colombia** (n=27): r=-0.49 — strong negative. Cooler Colombian farms produce better naturals. ✓
- **Brazil** (n=80): r=+0.26 — the *opposite*. Warmer Brazilian farms produce better naturals. ✗
- **Ethiopia** (n=17): r=+0.30 — also reversed, though small sample. ✗

**Washed and rainfall:**
- **Guatemala** (n=161): r=+0.11 — weakly positive. ✓
- **Honduras** (n=35): r=+0.20 — positive. ✓
- **Colombia** (n=121): r=-0.38 — strong *negative*. More rain = worse scores. ✗
- **Costa Rica** (n=45): r=-0.46 — also negative. ✗
- **Mexico** (n=198): r=+0.02 — no relationship at all. —

In [44]:
# Within-country: temperature vs score for naturals
naturals = df[df["processing_method"] == "Natural / Dry"].dropna(subset=["t2m", "total_cup_points"])
top_nat_countries = naturals["country_of_origin"].value_counts().head(4).index
nat_sub = naturals[naturals["country_of_origin"].isin(top_nat_countries)]

fig = px.scatter(
    nat_sub,
    x="t2m",
    y="total_cup_points",
    color="country_of_origin",
    trendline="ols",
    facet_col="country_of_origin",
    facet_col_wrap=2,
    labels={"t2m": "Temperature (°C)", "total_cup_points": "Cup Score"},
    hover_data=["farm_name", "region"],
    opacity=0.6,
)
fig.update_layout(height=500, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

In [45]:
# Within-country: rainfall vs score for washed
washed = df[df["processing_method"] == "Washed / Wet"].dropna(subset=["prectotcorr", "total_cup_points"])
top_wash_countries = ["Colombia", "Guatemala", "Mexico", "Costa Rica"]
wash_sub = washed[washed["country_of_origin"].isin(top_wash_countries)]

fig = px.scatter(
    wash_sub,
    x="prectotcorr",
    y="total_cup_points",
    color="country_of_origin",
    trendline="ols",
    facet_col="country_of_origin",
    facet_col_wrap=2,
    labels={"prectotcorr": "Rainfall (mm/day)", "total_cup_points": "Cup Score"},
    hover_data=["farm_name", "region"],
    opacity=0.6,
)
fig.update_layout(height=500, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

---

## 8. Soil — A Better Predictor Than Altitude

> **Observed in the data.** Soil data comes from [ISRIC SoilGrids](https://soilgrids.org/)
> at 250m resolution, matched to each farm's geocoded coordinates. The correlations
> and regression results are computed directly.

If altitude is a weak proxy for environmental conditions, what about the soil itself?
We pulled five soil properties for each farm location — pH, organic carbon density,
clay, sand, and silt content — averaged across the top 30cm (the coffee root zone).

**Soil alone explains 4.2% of score variance** — far more than altitude (0.02%).
Soil pH and sand/silt content are statistically significant predictors even after
controlling for country and processing method.

In [46]:
# Soil properties vs cup score
if has_soil:
    soil_cols = ["soil_ph", "soil_organic_carbon", "soil_clay_content", "soil_sand_content", "soil_silt_content"]
    soil_labels = ["pH", "Organic Carbon (hg/m³)", "Clay (g/kg)", "Sand (g/kg)", "Silt (g/kg)"]
    soil_df = df.dropna(subset=soil_cols + ["total_cup_points"])

    soil_corrs = soil_df[soil_cols + ["total_cup_points"]].corr()["total_cup_points"].drop("total_cup_points")
    soil_corrs.index = soil_labels

    # Include altitude for comparison
    alt_corr = df[["altitude_mean_meters", "total_cup_points"]].dropna().corr().iloc[0, 1]
    all_corrs = pd.concat([soil_corrs, pd.Series({"Altitude (m)": alt_corr})]).sort_values()
    colors = ["#d4a574" if abs(v) < 0.15 else "#8b5e3c" for v in all_corrs.values]

    fig = px.bar(
        x=all_corrs.values, y=all_corrs.index, orientation="h",
        text=[f"{v:.3f}" for v in all_corrs.values],
        labels={"x": "Correlation with Total Score (r)", "y": ""},
    )
    fig.update_traces(marker_color=colors)
    fig.update_layout(height=350, showlegend=False)
    fig.show()

In [47]:
# Soil pH and sand content vs cup score (the two strongest predictors)
if has_soil:
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        "Lower pH → higher scores (r=0.14, p=0.002)",
        "More sand → higher scores (r=0.11, p=0.005)",
    ])

    soil_scatter = df.dropna(subset=["soil_ph", "soil_sand_content", "total_cup_points"])

    fig.add_trace(go.Scatter(
        x=soil_scatter["soil_ph"], y=soil_scatter["total_cup_points"],
        mode="markers", marker=dict(color="#8b5e3c", opacity=0.3), showlegend=False,
    ), row=1, col=1)
    z = np.polyfit(soil_scatter["soil_ph"], soil_scatter["total_cup_points"], 1)
    x_line = np.linspace(soil_scatter["soil_ph"].min(), soil_scatter["soil_ph"].max(), 50)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode="lines", line=dict(color="#d94a4a", width=2), showlegend=False,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=soil_scatter["soil_sand_content"], y=soil_scatter["total_cup_points"],
        mode="markers", marker=dict(color="#8b5e3c", opacity=0.3), showlegend=False,
    ), row=1, col=2)
    z = np.polyfit(soil_scatter["soil_sand_content"], soil_scatter["total_cup_points"], 1)
    x_line = np.linspace(soil_scatter["soil_sand_content"].min(), soil_scatter["soil_sand_content"].max(), 50)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode="lines", line=dict(color="#d94a4a", width=2), showlegend=False,
    ), row=1, col=2)

    fig.update_xaxes(title_text="Soil pH", row=1, col=1)
    fig.update_xaxes(title_text="Sand Content (g/kg)", row=1, col=2)
    fig.update_yaxes(title_text="Total Cup Points", row=1, col=1)
    fig.update_layout(height=400)
    fig.show()

**Lower pH soils produce higher-scoring coffees** (coef=-0.64, p=0.002). This
aligns with coffee agronomy — Arabica thrives in slightly acidic soils (pH 4.5–6.0).
Higher sand and silt content also correlate positively with quality, likely because
well-drained soils stress the plant just enough to concentrate flavors in the cherry.

In a full regression model with country, processing, altitude, and soil together,
**soil adds 1.4 percentage points of R²** beyond what the other variables explain.
That's modest — but it's 70× more than altitude contributes on its own.

---

## 9. Variety — The Genetic Factor

> **Observed in the data.** Variety scores and regression results are computed
> directly from the CQI dataset. The interpretation of *why* certain varieties
> score higher involves inference about genetics and growing conditions.

Coffee variety (the cultivar or genetic lineage of the tree) is another factor
that could influence cup quality. With 1,300 samples spanning dozens of varieties,
we can test how much genetics matter — and whether certain varieties pair better
with certain processing methods.

In [48]:
# Top varieties by average score (min 15 samples)
variety_stats = (
    df.groupby("variety")
    .agg(avg_score=("total_cup_points", "mean"), count=("total_cup_points", "count"))
    .query("count >= 15")
    .sort_values("avg_score", ascending=True)
)

fig = px.bar(
    variety_stats.reset_index(),
    x="avg_score", y="variety", orientation="h",
    text=variety_stats.apply(lambda r: f"{r['avg_score']:.1f}  (n={int(r['count'])})", axis=1).values,
    labels={"avg_score": "Average Total Cup Points", "variety": ""},
    color_discrete_sequence=["#8b5e3c"],
)
fig.update_layout(xaxis=dict(range=[80, 86]), height=max(300, len(variety_stats) * 25), showlegend=False)
fig.show()

In [49]:
# Variety × processing interaction — which varieties benefit most from natural processing?
top_varieties = df["variety"].value_counts().head(8).index
vp = df[df["variety"].isin(top_varieties) & df["processing_method"].isin(["Washed / Wet", "Natural / Dry"])]
vp_stats = vp.groupby(["variety", "processing_method"])["total_cup_points"].agg(["mean", "count"]).reset_index()
vp_stats = vp_stats[vp_stats["count"] >= 5]

fig = px.bar(
    vp_stats, x="variety", y="mean", color="processing_method",
    barmode="group", text=vp_stats["mean"].round(1),
    labels={"mean": "Avg Cup Points", "variety": "", "processing_method": "Processing"},
    color_discrete_map=method_colors,
)
fig.update_layout(height=400)
fig.show()

**SL28 and SL34** (Kenyan selections) score highest among common varieties, consistent
with their reputation for complex, wine-like acidity. **Bourbon** varieties score
well and show the largest benefit from natural processing.

In multivariate regression, **variety explains about 9% of score variance** — far
more than processing (1.2%) or altitude (0.02%). But variety is confounded with
country (SL28 is almost exclusively Kenyan), so much of what "variety" captures
may really be origin effects.

**The interaction between variety and processing is real but small.** Some varieties
(Bourbon, Typica) show measurably higher scores when naturally processed, while
others (Caturra, Catuai) show little difference. This suggests fermentation's
effect on quality partly depends on the raw material — the genetic flavor potential
of the cherry.

---

## 10. The Rise of Experimental Processing

> **Observed in the data.** Processing method frequencies and scores come from
> 17,000+ Cup of Excellence competition entries (1999–2024). The trend toward
> experimental methods and their score advantage are directly measured.

The Cup of Excellence (CoE) is specialty coffee's most prestigious competition.
Over 25 years and 400+ competitions across 30 countries, it provides a window
into how processing methods are evolving — and whether newer, more controlled
fermentation techniques actually produce better cups.

Since ~2018, **experimental methods** — anaerobic fermentation, carbonic maceration,
lactic acid inoculation, wild yeast processing — have surged from near-zero to
10–20% of competition entries.

In [50]:
# Experimental processing methods over time (Cup of Excellence data)
if has_coe:
    def classify_process(p):
        if pd.isna(p) or str(p).strip() == "":
            return None  # exclude unknowns from this chart
        p = str(p).lower()
        if any(k in p for k in ["anaerobic", "anaerobico", "anaeróbico"]):
            return "Anaerobic"
        if any(k in p for k in ["carbonic", "maceracion carbonica"]):
            return "Carbonic Maceration"
        if "lactic" in p:
            return "Lactic Fermentation"
        if "wild yeast" in p:
            return "Wild Yeast"
        if "experimental" in p or "exerimental" in p or "alchemy" in p:
            return "Experimental (other)"
        if any(k in p for k in ["honey", "semi-washed", "semi washed", "semiwashed"]):
            return "Honey"
        if any(k in p for k in ["natural", "dry"]):
            return "Natural"
        if any(k in p for k in ["washed", "lavado", "wet process", "depulped", "demucilaged"]):
            return "Washed"
        return "Other"

    coe["process_category"] = coe["process"].apply(classify_process)
    coe_known = coe.dropna(subset=["process_category"])
    coe_known["year"] = pd.to_numeric(coe_known["year"], errors="coerce")
    coe_known = coe_known[coe_known["year"] >= 2007]

    experimental = ["Anaerobic", "Carbonic Maceration", "Lactic Fermentation", "Wild Yeast", "Experimental (other)"]
    coe_known["is_experimental"] = coe_known["process_category"].isin(experimental)

    by_year = coe_known.groupby("year").agg(
        total=("score", "count"),
        experimental=("is_experimental", "sum"),
    ).reset_index()
    by_year["pct_experimental"] = by_year["experimental"] / by_year["total"] * 100

    fig = px.bar(
        by_year, x="year", y="pct_experimental",
        text=by_year["pct_experimental"].round(1).astype(str) + "%",
        labels={"year": "Year", "pct_experimental": "% of Entries Using Experimental Methods"},
        color_discrete_sequence=["#8b5e3c"],
    )
    fig.update_layout(height=350, showlegend=False, xaxis=dict(dtick=1))
    fig.show()

In [51]:
# Average scores by processing category (Cup of Excellence)
if has_coe:
    cat_stats = (
        coe_known.groupby("process_category")["score"]
        .agg(["mean", "count"])
        .query("count >= 5")
        .sort_values("mean", ascending=True)
        .reset_index()
    )

    colors = ["#d94a4a" if cat in experimental else "#8b5e3c"
              for cat in cat_stats["process_category"]]

    fig = px.bar(
        cat_stats, x="mean", y="process_category", orientation="h",
        text=cat_stats.apply(lambda r: f"{r['mean']:.2f}  (n={int(r['count'])})", axis=1),
        labels={"mean": "Avg CoE Score", "process_category": ""},
    )
    fig.update_traces(marker_color=colors)
    fig.update_layout(height=350, showlegend=False, xaxis=dict(range=[86, 90]))
    fig.show()

Experimental methods (shown in red) score **0.5–0.7 points higher** than traditional
washed or natural processing in Cup of Excellence competitions. Carbonic maceration
leads at 89.1, followed by anaerobic fermentation at 88.6.

This is a meaningful gap in a competition where the difference between 1st and 10th
place is often less than 2 points. But there are important caveats:

- **Selection bias**: farmers using experimental methods are often the most skilled
  and invested. The technique may not be the cause — the farmer's overall approach might be.
- **Novelty bias**: competition judges may unconsciously reward unusual flavor profiles.
- **The trend is accelerating**: experimental methods went from 0% of entries before
  2018 to nearly 20% by 2024, suggesting the industry believes these techniques work.

What's clear is that **controlled fermentation is the frontier of coffee quality**.
Rather than leaving fermentation to ambient conditions, producers are engineering it —
controlling temperature, oxygen, duration, and even inoculating with specific microbes.
This is the strongest signal yet that fermentation isn't just a post-harvest chore
but a precision tool for shaping cup quality.

---

## What This Means

> Items marked **observed** are directly from the data. Items marked **inferred**
> are interpretations that go beyond what the data alone proves.

1. **Flavor and aftertaste drive quality** (observed). Body is the most
   independent attribute and the one that differs most across processing methods.

2. **More fermentation correlates with higher scores** (observed) — but with
   more risk. Natural and honey processing edge ahead of washed on average,
   but their variance is wider.

3. **Altitude is a weak predictor** (observed, r=0.11). **Soil is 70× better**
   (R²=4.2% vs 0.02%). Lower pH and well-drained (sandy) soils correlate with
   higher quality, consistent with coffee agronomy.

4. **Variety matters more than processing** (observed). Variety explains ~9%
   of score variance vs 1.2% for processing. But variety is heavily confounded
   with country of origin.

5. **Country is the strongest single factor** (observed). In a full model with
   processing, variety, altitude, soil, and climate, country dummies contribute
   the most explanatory power (R²≈0.25). Country is a proxy for unmeasured
   variables — farming tradition, genetic selection, terroir, infrastructure.

6. **Experimental processing methods score higher in competitions** (observed,
   Cup of Excellence data). Anaerobic, carbonic maceration, and controlled
   fermentation coffees average 0.5–0.7 points above traditional methods.
   Whether this reflects technique, farmer skill, or novelty bias is unclear.

7. **The aggregate climate-processing correlations are real but misleading**
   (observed + inferred). Within-country analysis shows the patterns are
   inconsistent, driven partly by country composition.

### The bigger picture

Coffee quality emerges from the interaction of genetics (variety), environment
(soil, climate, altitude), and human decisions (processing, farming practice).
No single variable dominates. The most promising frontier is **controlled
fermentation** — where producers move from passive environmental exposure to
active management of the microbial process that transforms a cherry into a
cup of coffee.

---

## Data Sources

This analysis draws on four independent datasets, merged by geocoded farm coordinates:

| Dataset | Source | What it provides | Records |
|---------|--------|-----------------|---------|
| **CQI Arabica Database** | [jldbc/coffee-quality-database](https://github.com/jldbc/coffee-quality-database) (GitHub) | Cupping scores, processing method, variety, altitude, origin for CQI-graded coffees | 1,311 coffees |
| **NASA POWER** | [power.larc.nasa.gov](https://power.larc.nasa.gov/) | Monthly temperature, humidity, rainfall, and solar radiation at farm coordinates, filtered to each country's harvest window | 337,610 monthly records across 265 locations |
| **ISRIC SoilGrids** | [soilgrids.org](https://soilgrids.org/) ([REST API](https://rest.isric.org/soilgrids/v2.0/docs) / [WCS](https://maps.isric.org/mapserv)) | Soil pH, organic carbon, clay, sand, and silt content at 250m resolution, averaged across the top 30cm (coffee root zone) | 3,975 soil records across 265 locations |
| **Cup of Excellence** | [cupofexcellence.org](https://cupofexcellence.org/competition-auction-results/) / [allianceforcoffeeexcellence.org](https://allianceforcoffeeexcellence.org/competition-auction-results/) | Competition scores, processing methods, varieties, regions, and farm names from 25 years of CoE competitions | 17,211 entries across 400+ competitions (1999–2026) |

**Geocoding:** Farm regions from the CQI dataset were geocoded to latitude/longitude using [OpenStreetMap Nominatim](https://nominatim.openstreetmap.org/) via the [geopy](https://geopy.readthedocs.io/) library (362 unique locations resolved).

**Harvest windows:** Climate data was filtered to country-specific harvest months using a manually compiled calendar of coffee harvest seasons for 32 producing countries.

Source code: [github.com/jsdrews/coffee-data](https://github.com/jsdrews/coffee-data)